In [1]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, EarlyStoppingCallback
from transformers import pipeline
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
import numpy as np
import torch
from torch import nn
import json
from collections import Counter

Загрузим датасет

In [2]:
tags_to_names = {
    "cs": "Computer Science",
    "econ": "Economics",
    "eess": "Electrical Engineering and Systems Science",
    "math": "Mathematics",
    "physics": "Physics",
    "q-bio": "Quantitive Biology",
    "q-fin": "Quantitive Finance",
    "stat": "Statistics"
}

In [3]:
ds_balanced = load_dataset("pinmax/arxiv_balanced_soft_labels")

Скачаем уже предобученную модель scibert с HF

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("allenai/scibert_scivocab_uncased")
model = AutoModelForSequenceClassification.from_pretrained("allenai/scibert_scivocab_uncased", num_labels=len(tags_to_names))

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Токенизируем abstract и title в датасете

In [5]:
def tokenize_function(examples):
    return tokenizer(examples["title"], examples["abstract"], padding="max_length", truncation=True, max_length=512)

In [6]:
ds_balanced = ds_balanced.map(tokenize_function, keep_in_memory=True)

Map:   0%|          | 0/139511 [00:00<?, ? examples/s]

Map:   0%|          | 0/34878 [00:00<?, ? examples/s]

In [7]:
ds_balanced.set_format("torch", columns=["input_ids", "attention_mask", "token_type_ids", "label"])

In [8]:
ds_balanced

DatasetDict({
    train: Dataset({
        features: ['title', 'abstract', 'label', 'soft_labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 139511
    })
    test: Dataset({
        features: ['title', 'abstract', 'label', 'soft_labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 34878
    })
})

In [ ]:
from huggingface_hub import login

login("")

In [25]:
TrainingArguments?

Дообучим модель

In [54]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds, average="weighted"),
        "precision": precision_score(labels, preds, average="weighted", zero_division=0),
        "recall": recall_score(labels, preds, average="weighted", zero_division=0),
    }

args = TrainingArguments(
    output_dir="./article-classifier",
    num_train_epochs=5,
    learning_rate=2e-5,
    lr_scheduler_type="cosine",

    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    fp16=True,

    warmup_ratio=0.1,
    label_smoothing_factor=0.1,

    logging_strategy="epoch",
    logging_steps=1,
    save_strategy="epoch",
    save_total_limit=2,
    eval_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,

    push_to_hub=True,
    hub_model_id="pinmax/article-classifier-scibert-big-dataset",

    disable_tqdm=False,
    dataloader_num_workers=2,

    optim="adamw_torch",
    weight_decay=0.1,
    use_cpu=False,
)



warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [55]:
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=ds_balanced["train"],
    eval_dataset=ds_balanced["test"],
    compute_metrics=compute_metrics,
)

In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.781486,0.758256,0.870090,0.869158,0.873496,0.870090
2,0.690574,0.741516,0.879150,0.879216,0.881152,0.879150


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]